# 🧹 Lekcja 3: Dziury w liście pasażerów (`NaN`), tajemnica kabin (`Has_Cabin`) i zamiana słów na liczby

Zanim zbudujemy pierwszy model klasyfikacyjny dla Titanica, musimy rozwiązać dokładnie te same dwa problemy techniczne, które spotkaliśmy przy cenach domów:
1. **Modele Machine Learningu nie tolerują pustych kratek (`NaN`)** — jedno brakujące pole w wieku lub cenie biletu zatrzyma algorytm z błędem.
2. **Modele nie rozumieją angielskich słów** takich jak `'male'`, `'female'` czy nazw portów `'S'`, `'C'`, `'Q'` — musimy przetłumaczyć je na liczby!

Zacznijmy od sprawdzenia na wykresie, gdzie dokładnie brakuje danych w obu naszych plikach: `../data/train.csv` oraz `../data/test.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Wczytujemy oba zbiory danych z folderu ../data/
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

# 2. Liczymy puste kratki (NaN) w obu tabelach
braki_train = train.isnull().sum()
braki_test = test.isnull().sum()

print('--- Braki (NaN) w zbiorze treningowym (train.csv — 891 pasażerów) ---')
print(braki_train[braki_train > 0])

print('\n--- Braki (NaN) w zbiorze testowym (test.csv — 418 pasażerów) ---')
print(braki_test[braki_test > 0])

# 3. Tworzymy cechę Has_Cabin (Czy pasażer miał przypisany numer kabiny? 1 = Tak, 0 = Nie)
train['Has_Cabin'] = train['Cabin'].notnull().astype(int)
test['Has_Cabin'] = test['Cabin'].notnull().astype(int)

przezycie_kabina = (train.groupby('Has_Cabin')['Survived'].mean() * 100).round(1)

# 4. WIZUALIZACJA: Po lewej braki danych, po prawej odkrycie związane z kabiną (Has_Cabin)!
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Wykres lewy: Liczba braków w train.csv
tylko_braki_tr = braki_train[braki_train > 0].sort_values(ascending=False)
ax1.bar(tylko_braki_tr.index, tylko_braki_tr.values, color=['salmon', 'skyblue', 'gold'], edgecolor='black', width=0.45)
ax1.axhline(891, color='red', linestyle='--', linewidth=1.5, label='Wszyscy pasażerowie w train (891)')
ax1.set_title('1. Brakujące dane (NaN) w pliku train.csv')
ax1.set_ylabel('Liczba pustych komórek (NaN)')
ax1.set_ylim(0, 1000)
for i, v in enumerate(tylko_braki_tr.values):
    ax1.text(i, v + 20, f'{v} ({v/len(train)*100:.1f}%)', ha='center', fontweight='bold')
ax1.legend()
ax1.grid(axis='y', linestyle=':', alpha=0.6)

# Wykres prawy: Czy posiadanie wpisanej kabiny (Has_Cabin) wpływało na przeżycie?
etykiety_kabin = ['0: Brak numeru kabiny\n(687 pasażerów)', '1: Ma wpisaną kabinę\n(204 pasażerów)']
ax2.bar(etykiety_kabin, przezycie_kabina.values, color=['lightgray', 'mediumseagreen'], edgecolor='black', width=0.45)
ax2.set_title('2. Szansa przeżycia a posiadanie numeru kabiny (Has_Cabin)')
ax2.set_ylabel('Procent ocalałych (%)')
ax2.set_ylim(0, 100)
for i, v in enumerate(przezycie_kabina.values):
    ax2.text(i, v + 2.5, f'{v}%', ha='center', fontweight='bold', fontsize=12)
ax2.grid(axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

### 🕵️‍♂️ Jak zamienić 77% braków w kolumnie `Cabin` w potężną wskazówkę dla modelu?

Spójrz na oba wykresy powyżej:
1. W kolumnie **`Cabin`** brakuje aż **687 na 891 wpisów (`77.1%`)**. Gdybyśmy próbowali zgadywać konkretne numery kabin (np. `'C85'`), ponieślibyśmy klęskę. Ale zamiast wyrzucać kolumnę `Cabin` do kosza bez zastanowienia, zadajmy pytanie detektywa: **„Dlaczego jedni pasażerowie mają w dokumentach numer kabiny, a inni mają tam `NaN`?”**
2. Odpowiedź kryje się na **prawym wykresie**: wśród 204 pasażerów, którzy mieli przypisany numer prywatnej kabiny (**`Has_Cabin = 1`**), przeżyło aż **`66.7%` osób**! Natomiast wśród pasażerów bez numeru kabiny (**`Has_Cabin = 0`**) przeżyło tylko **`30.0%`**!

#### Jak działa linijka `train['Cabin'].notnull().astype(int)`?
* **`.notnull()`** — to dokładne przeciwieństwo `.isnull()`. Zadaje pytanie: *„Czy kratka NIE JEST pusta?”* (`True` = jest wpisana kabina, `False` = jest `NaN`).
* **`.astype(int)`** — zamienia słowa logiczne na czyste liczby całkowite (`int` od *integer*): `True` zamienia na **`1`**, a `False` na **`0`**!

---

### 🧭 Plan naprawy pozostałych 3 kolumn z brakami (`Age`, `Embarked`, `Fare`)

| Kolumna z brakami | Ile brakuje w `train` / `test`? | Co to za dane? | Jak je uzupełniamy i dlaczego? |
| :--- | :---: | :--- | :--- |
| **`Age` (Wiek)** | `177` w `train`<br>`86` w `test` | Liczba ciągła (lata) | Uzupełniamy **medianą wieku z `train` (`28.0` lat)**. *(W Lekcji 5 pokażemy jeszcze sprytniejszy sposób — osobną medianę wieku dla chłopców, panien i dorosłych!)*. |
| **`Embarked` (Port)** | `2` w `train`<br>`0` w `test` | Tekst (`'S'`, `'C'`, `'Q'`) | Brakuje tylko u 2 pasażerek. Uzupełniamy **dominantą/modą (`.mode()[0]`)**, czyli najpopularniejszym portem **`'S'` (Southampton)**, w którym wsiadło aż 72% pasażerów. |
| **`Fare` (Cena biletu)** | `0` w `train`<br>**`1` w `test`!** | Liczba ciągła (funty £) | Uwaga na pułapkę: w `train.csv` kolumna `Fare` była w 100% czysta, ale w `test.csv` jeden starszy pasażer 3. klasy ma tam `NaN`! Uzupełniamy **medianą ceny biletu (`14.45 £`)** (bo średnia jest zawyżona przez milionerów!). |

In [ ]:
# 1. ŁATANIE BRAKÓW LICZBOWYCH (Age oraz Fare) — używamy mediany wyliczonej na zbiorze treningowym!
mediana_wieku = train['Age'].median()
mediana_biletu = train['Fare'].median()

train['Age'] = train['Age'].fillna(mediana_wieku)
test['Age'] = test['Age'].fillna(mediana_wieku)

test['Fare'] = test['Fare'].fillna(mediana_biletu)

# 2. ŁATANIE BRAKÓW TEKSTOWYCH (Embarked) — używamy najczęstszego portu (.mode()[0] -> 'S')
najczestszy_port = train['Embarked'].mode()[0]
train['Embarked'] = train['Embarked'].fillna(najczestszy_port)
test['Embarked'] = test['Embarked'].fillna(najczestszy_port)

# 3. KODOWANIE TEKSTU NA LICZBY:
# A) Płeć (Sex): zamieniamy 'male' -> 0, 'female' -> 1 za pomocą słownika i .map()
mapa_plci = {'male': 0, 'female': 1}
train['Sex_Num'] = train['Sex'].map(mapa_plci)
test['Sex_Num'] = test['Sex'].map(mapa_plci)

# B) Port (Embarked): rozbijamy na 3 kolumny-przełączniki 0/1 za pomocą One-Hot Encodingu (pd.get_dummies)
porty_ohe = pd.get_dummies(train['Embarked'], prefix='Port', dtype=int)

print(f'Uzupełniono braki wieku medianą: {mediana_wieku} lat')
print(f'Uzupełniono brak ceny biletu w test.csv medianą: {mediana_biletu:.2f} £')
print(f'Uzupełniono 2 braki portu najczęstszą wartością: {najczestszy_port} (Southampton)\n')

print('--- Podgląd zakodowanych cech dla pierwszych 5 pasażerów ---')
podglad = pd.concat([train[['Name', 'Sex', 'Sex_Num', 'Age', 'Has_Cabin', 'Embarked']], porty_ohe], axis=1)
podglad.head(5)

### 🔍 Rozbiórka kodu czyszczącego i kodującego tekst

1. **Dlaczego przy uzupełnianiu `test['Age']` użyliśmy `mediana_wieku` policzonej z `train['Age']`?**  
   W prawdziwym świecie zbiór treningowy (`train`) to nasza jedyna baza wiedzy historycznej. Kiedy przychodzi nowy pasażer do oceny (ze zbioru `test`), wszelkie statystyki pomocnicze (takie jak typowy wiek czy typowa cena biletu) bierzemy z naszej bazy treningowej!
2. **Kodowanie binarne płci (`{'male': 0, 'female': 1}`):**  
   Ponieważ kolumna `Sex` ma tylko **2 kategorie**, nie musimy tworzyć dwóch osobnych kolumn — wystarczy jeden przełącznik `0` (mężczyzna) lub `1` (kobieta).
3. **One-Hot Encoding dla portu (`pd.get_dummies(train['Embarked'], prefix='Port', dtype=int)`):**  
   Porty zaokrętowania są trzy (`C`, `Q`, `S`) i nie tworzą żadnej drabiny jakości. Parametr `prefix='Port'` dodaje czytelny przedrostek, tworząc trzy kolumny zero-jedynkowe: `Port_C`, `Port_Q` oraz `Port_S`.

In [ ]:
# 🎯 ĆWICZENIE 1: Sprawdźmy, w której klasie biletu (Pclass: 1, 2 czy 3)
# pasażerowie najczęściej mieli przypisany numer kabiny (Has_Cabin == 1)!
#
# 1. Użyj na tabeli `train` metody: train.groupby('Pclass')['Has_Cabin'].mean() * 100
# 2. Wyświetl wynik za pomocą print() — jaki procent pasażerów 1. klasy miał znany numer kabiny,
#    a jaki procent w 3. klasie?

# Miejsce na Twój kod:


In [ ]:
# 🎯 ĆWICZENIE 2: Kontrola czystości naszych kolumn przed budową modelu!
#
# Przygotowaliśmy zestaw 5 czystych, liczbowych cech: ['Pclass', 'Sex_Num', 'Age', 'Fare', 'Has_Cabin']
# Sprawdź za pomocą .isnull().sum(), czy w tabeli `train` oraz w tabeli `test`
# dla tych 5 kolumn liczba braków (NaN) wynosi dokładnie 0!
#
# Podpowiedź:
# wybrane_cechy = ['Pclass', 'Sex_Num', 'Age', 'Fare', 'Has_Cabin']
# print(train[wybrane_cechy].isnull().sum())
# print(test[wybrane_cechy].isnull().sum())

# Miejsce na Twój kod:


---
### 📌 Podsumowanie lekcji w 3 punktach

1. **Zamiana braków w `Cabin` na nową cechę `Has_Cabin`:** Zamiast zgadywać brakujące w 77% numery kabin, stworzyliśmy przełącznik `Has_Cabin` (`1` = ma wpisaną kabinę, `0` = brak), odkrywając, że osoby z przypisaną kabiną przeżyły aż w `66.7%` (wobec `30.0%` bez kabiny).
2. **Statystyczne łatanie dziur (`Age`, `Fare`, `Embarked`):** Brakujące liczby w wieku (`Age`) i cenie biletu (`Fare` w `test.csv`) uzupełniliśmy odporną na bogaczy **medianą** (`28.0` lat oraz `14.45 £`), a 2 brakujące porty (`Embarked`) najczęstszą wartością **`.mode()[0]` (`'S'`)**.
3. **Kodowanie tekstu na liczby:** Kolumnę dwuwartościową `Sex` zamieniliśmy słownikiem `.map({'male': 0, 'female': 1})` na jedną kolumnę zero-jedynkową, a trzy porty `Embarked` rozbiliśmy na przełączniki `0`/`1` za pomocą `pd.get_dummies(..., dtype=int)`.